<a href="https://colab.research.google.com/github/matheus-pvds/TIA_FILES/blob/main/Trabalho_final_de_AMLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TRABALHO FINAL DE APLICAÇÕES DE MODELOS NATURAIS DE LINGUAGEM

##SETUP INICIAL & IMPORTAÇÕES

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
URL = "https://www.ifnmg.edu.br/cursos-ofertados/531-cursos-ofertados-cead/cursos-tecnicos-cead/38718-curso-tecnico-em-inteligencia-artificial"
HTML_PATH = "/content/drive/MyDrive/DOCS/Portal IFNMG.html"
PDF_PATH = "/content/drive/MyDrive/DOCS/PDFS/Versão Final PPC Técnico em IA CEADi-CONSUP.pdf"

In [ ]:
import requests
import os

In [ ]:
with open(HTML_PATH, 'r', encoding='utf-8', errors='replace') as f:
    html_local = f.read()

import bs4

soup = bs4.BeautifulSoup(html_local, 'html.parser')
print(soup)

<!DOCTYPE html>

<!-- saved from url=(0135)https://www.ifnmg.edu.br/cursos-ofertados/531-cursos-ofertados-cead/cursos-tecnicos-cead/38718-curso-tecnico-em-inteligencia-artificial -->
<html class="no-js" dir="ltr" lang="pt-br" style="height: 100%;"><!--<![endif]--><head><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<!-- Google tag (gtag.js) -->
<script async="" src="./Portal IFNMG - Curso Técnico em Inteligência Artificial_files/ga.js.download" type="text/javascript"></script><script async="" src="./Portal IFNMG - Curso Técnico em Inteligência Artificial_files/js"></script>
<script>
        window.dataLayer = window.dataLayer || [];
        function gtag(){dataLayer.push(arguments);}
        gtag('js', new Date());

        gtag('config', 'G-E1BL1MY6GK');
        </script>
<!--[if lt IE 9]>
    <script src="/templates/padraogoverno01/js/html5shiv.js"></script>
    <![endif]-->
<!--[if lt IE 10]>
    <link rel="stylesheet" href="/templates/padraogoverno01/css/ie.cs

In [ ]:
tg_div = soup.select_one('#content-section > div > div.item-page')
texto = tg_div.get_text(separator=" ")
format_txt = f"{texto[:47]}\n{texto[190:-2007]}"
print(format_txt)

##1. MODELO DE GUARDRAIL CONTRA INJEÇÃO

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client.models import Distance

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

model = AutoModelForSequenceClassification.from_pretrained("cguardion/ModernGuard-1")
tokenizer = AutoTokenizer.from_pretrained("guardion/ModernGuard-1")

device = "cuda" if torch.cuda.is_available() else "cpu"

def check_injection(text: str) -> str:

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048)
    outputs = model(**inputs).to(device)
    probs = outputs.logits.softmax(dim=-1)

    threshold = 0.5
    is_injection = probs[0, 1] >= threshold
    score = probs[0, 1].item()

    return 'INJECTION' if is_injection else 'SAFE'

def ingestion(text: str) -> str:

    pass


def _():

    pass



Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]